# 02 · Pipeline RAG passo a passo

O enunciado pede para **demonstrar** cada etapa: transformação da pergunta, recuperação semântica, seleção dos conteúdos mais relevantes, construção do contexto e geração da resposta. Este notebook roda uma pergunta por cada etapa e mostra o que entra e o que sai.

As etapas 1 (reescrita) e 5 (geração) usam o LLM local (Qwen2.5 7B via Ollama) e só rodam com o Ollama no ar. Sem ele, o notebook mostra a recuperação, que também é local.

In [1]:
import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)

from src import config
from src.generation import llm
from src.indexing.embeddings import codificar_consultas
from src.retrieval.analise_pergunta import analisar_com_llm, analisar_por_regras
from src.retrieval.busca import buscar, rrf
from src.retrieval.indice import carregar

idx = carregar()
PERGUNTA = "O que os clientes relatam sobre problemas relacionados à entrega?"
print(f"{len(idx.docs):,} documentos | vetores {idx.vetores.shape} | LLM disponível: {llm.disponivel()}")

36,038 documentos | vetores (36038, 768) | LLM disponível: True


## 1. Processamento da pergunta

O gestor pergunta com vocabulário de gestor ("problemas relacionados à entrega"); o cliente escreve "não chegou", "atrasou", "até hoje nada". O LLM decide o escopo e o tipo de pergunta e gera reescritas no vocabulário do cliente; os filtros de metadados vêm de regras ancoradas no texto (decisão D14 do README).

In [2]:
analise = analisar_com_llm(PERGUNTA, idx.categorias) if llm.disponivel() else analisar_por_regras(PERGUNTA)
print("origem:", analise.origem, "| escopo:", analise.no_escopo, "| tipo:", analise.tipo, "| tema:", analise.tipo_tema)
print("filtros:", analise.filtros.descrever())
for c_ in analise.consultas: print(" -", c_)

origem: llm | escopo: True | tipo: exploratoria | tema: problema
filtros: sem filtros
 - O que os clientes relatam sobre problemas relacionados à entrega?
 - o produto demorou muito para chegar
 - não recebi o produto no prazo prometido
 - a entrega foi cancelada sem aviso


## 2. Representação vetorial

A pergunta vira um vetor de 768 dimensões com o mesmo modelo que indexou as avaliações (`multilingual-e5-base`, prefixo `query:`). Como os vetores são normalizados, o produto interno é a similaridade de cosseno.

In [3]:
v = codificar_consultas([PERGUNTA])[0]
sims = idx.vetores @ v
top = np.argsort(-sims)[:8]
pd.DataFrame({"similaridade": sims[top].round(3), "nota": idx.docs.loc[top, "nota"].values, "texto": idx.docs.loc[top, "texto"].values})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,similaridade,nota,texto
0,0.864,1,"Em partes. Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, p..."
1,0.859,1,ainda não recebi o meu produto quando vão me entregar ? ou meu dinheiro de volta
2,0.859,1,TERRIVEL ESTE PROBLEMA DA ENTREGA
3,0.857,4,Boa. Fatisfeito com o produto é a entrega
4,0.855,2,Como podem mandar um produto errado ? Não existe conferências? Loja falha
5,0.855,2,Produto não entregue. Gostaria de informações sobre a não entrega do produto.
6,0.855,1,Como resolver esta situação q o produto ainda não foi entregue?
7,0.854,1,Produto não entregue. O produto ainda não foi entregue. Recebi a mensagem de que houve dificuldade na entrega


## 3. Recuperação híbrida e fusão

Busca densa (sentido) e BM25 (palavras exatas) trazem candidatos diferentes. A Reciprocal Rank Fusion soma `1/(60 + posição)` de cada lista: quem aparece bem nas duas sobe.

In [4]:
permitidos = np.arange(len(idx.docs))
densa = idx.busca_densa(v, permitidos, 100)
bm25 = idx.busca_bm25(PERGUNTA, permitidos, 100)
fundido = rrf([densa, bm25])
so_densa = {p for p, _ in densa} - {p for p, _ in bm25}
so_bm25 = {p for p, _ in bm25} - {p for p, _ in densa}
print(f"densa: {len(densa)} | bm25: {len(bm25)} | nas duas: {len({p for p,_ in densa} & {p for p,_ in bm25})} | após fusão: {len(fundido)}")
print("\nSó o BM25 achou (palavra exata):")
for p in list(so_bm25)[:3]: print(" -", idx.docs.at[p, "texto"][:120])
print("\nSó a busca densa achou (sentido parecido, outras palavras):")
for p in list(so_densa)[:3]: print(" -", idx.docs.at[p, "texto"][:120])

densa: 100 | bm25: 100 | nas duas: 9 | após fusão: 191

Só o BM25 achou (palavra exata):
 - Eu pedi a devolução do dinheiro, após inúmeras tentativas de contato sobre a entrega e nem sobre isso obtive resposta.
 - Compra tranquila e entrega sem problemas.
 - Recebido rapidamente e sem problemas

Só a busca densa achou (sentido parecido, outras palavras):
 - TERRIVEL ESTE PROBLEMA DA ENTREGA
 - Não tive problemas com a entrega nem com o produto, o problema foi com o pagamento que passou 2 vezes no meu cartao, fiz
 - Produto não foi entregue. Gostaria de saber como vai ficar minha situação, pois o produto não foi entague


## 4. Seleção: reranking, limiar e diversidade

O cross-encoder lê pergunta e avaliação juntas e dá um score de relevância. Avaliações abaixo do limiar calibrado (`reports/calibracao_limiar.json`) não contam como evidência; com menos de 3 acima dele, o sistema recusa. O MMR evita 15 variações da mesma frase.

In [5]:
r = buscar(analise.consultas, analise.filtros)
print(f"universo {r.universo:,} | candidatos fundidos {r.candidatos_fundidos} | abaixo do limiar {r.descartadas_limiar} | suficiente: {r.suficiente}")
pd.DataFrame([e.como_dict() for e in r.evidencias])[["score_rerank", "nota", "status_entrega", "categoria", "texto"]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

universo 36,038 | candidatos fundidos 817 | abaixo do limiar 21 | suficiente: True


,score_rerank,nota,status_entrega,categoria,texto
0,10.038,3,atrasado,moveis decoracao,Não recebi o produto no prazo prometido.
1,8.813,3,no_prazo,informatica acessorios,Produto demorou muito pra chegar.
2,8.679,5,no_prazo,utilidades domesticas,O produto demorou muito pra chegar.
3,7.823,2,no_prazo,utilidades domesticas,O produto demorou muito p chegar.
4,7.344,1,atrasado,relogios presentes,O produto não chegou no prazo prometido.
5,6.343,2,atrasado,ferramentas jardim,Produto não entregue. O produto não foi entregue no prazo estipulado pelo vendedor.
6,5.756,2,atrasado,beleza saude,Produto demorou muito para chegar
7,5.719,3,no_prazo,eletrodomesticos,Demorou muito para chegar o produto
8,4.943,4,no_prazo,relogios presentes,O produto demorou muito para chegar
9,4.579,5,no_prazo,utilidades domesticas,PRODUTO OK APENAS DEMOROU MUITO PRA CHEGAR


## 5. Contexto e geração

O contexto numera as avaliações [R1]..[Rn] com seus metadados. O prompt exige citação em cada afirmação e proíbe números que não venham da tabela de frequência. Depois da geração, o código remove citações para avaliações que não estão no contexto.

In [6]:
from src.generation.resposta import gerar, montar_contexto
print(montar_contexto(PERGUNTA, analise.filtros.descrever(), r.evidencias, None)[:2500])

Pergunta do gestor: O que os clientes relatam sobre problemas relacionados à entrega?

Filtros aplicados: sem filtros

Avaliações recuperadas:
[R1] nota 3 | moveis decoracao | entregue com atraso | compra 2017-09-09
"Não recebi o produto no prazo prometido."
[R2] nota 3 | informatica acessorios | entregue no prazo | compra 2018-01-25
"Produto demorou muito pra chegar."
[R3] nota 5 | utilidades domesticas | entregue no prazo | compra 2018-03-29
"O produto demorou muito pra chegar."
[R4] nota 2 | utilidades domesticas | entregue no prazo | compra 2017-10-25
"O produto demorou muito p chegar."
[R5] nota 1 | relogios presentes | entregue com atraso | compra 2018-02-08
"O produto não chegou no prazo prometido."
[R6] nota 2 | ferramentas jardim | entregue com atraso | compra 2018-03-30
"Produto não entregue. O produto não foi entregue no prazo estipulado pelo vendedor."
[R7] nota 2 | beleza saude | entregue com atraso | compra 2017-11-07
"Produto demorou muito para chegar"
[R8] nota 3 | elet

In [7]:
if llm.disponivel():
    resp = gerar(PERGUNTA, analise.filtros.descrever(), r.evidencias, None)
    print(resp.texto)
    print("\ncitações válidas:", resp.citacoes_validas, "| inválidas removidas:", resp.citacoes_invalidas)
    print("lacunas:", resp.lacunas)
else:
    print("Ollama fora do ar: geração não executada. Exemplos gerados em reports/exemplos_respostas.json.")

Os clientes relatam atrasos na entrega e falta de comunicação, o que resulta em notas baixas.

- Muitos clientes relataram atrasos na entrega. [R1][R5][R6][R7][R8][R9][R13]
- Algumas reclamações mencionam a falta de comunicação sobre atrasos. [R14]
- Há casos de produtos que não foram entregues. [R11][R15]
- Os clientes que relataram atrasos geralmente deram notas baixas. [R1][R5][R6][R7][R8][R9][R11][R12][R13][R14][R15]
- Os produtos que chegaram atrasados foram classificados como 'entregue com atraso' ou 'entregue no prazo', mas com atraso percebido pelo cliente. [R1][R5][R6][R7][R8][R9][R13]

citações válidas: 28 | inválidas removidas: []
lacunas: Não há informações sobre a eficiência do serviço de entrega ou a razão das demoras.


## 6. Quando o sistema deve dizer que não sabe

In [8]:
from src.pipeline import responder
for q in ["Qual foi o faturamento da Olist em 2017?", "O que os clientes acham das entregas feitas por drone?"]:
    res = responder(q)
    print(f"{res.status:24s} max_score={res.busca.max_score if res.busca else None}  {q}\n   -> {res.mensagem}\n")

fora_do_escopo           max_score=None  Qual foi o faturamento da Olist em 2017?
   -> Esta pergunta está fora do que a base de avaliações de clientes consegue responder. A pergunta busca informações sobre o faturamento da empresa em um ano específico. (assunto: dados_da_empresa)



evidencia_insuficiente   max_score=-1.2357964515686035  O que os clientes acham das entregas feitas por drone?
   -> Não encontrei avaliações suficientes para responder com segurança (1 avaliações relevantes; o mínimo é 3). Tente ampliar os filtros ou reformular a pergunta.

